# 🛡️ Error Handling and Reliability Patterns

## Learning Objectives
In this notebook, you will learn:
1. **Retry with backoff** - wrap unreliable calls in exponential-backoff retry logic with jitter.
2. **Circuit breaker** - stop hammering a failing service and let it recover before retrying.
3. **Fallback chains** - cascade across multiple models/providers, with response caching.
4. **Graph-level resilience** - build a LangGraph agent whose own nodes retry and gracefully report errors.

## Prerequisites
- Familiarity with LangGraph `StateGraph`, nodes, and conditional edges (see prior notebooks in this phase)
- `OPENAI_API_KEY` and `ANTHROPIC_API_KEY` set in a `.env` file at the project root
- Packages: `langchain-openai`, `langchain-anthropic`, `langgraph`, `langsmith`, `python-dotenv`

> **Note**: Several demos in this notebook *intentionally* raise or simulate exceptions (random failures, a circuit that trips open, a node that returns an error state) to show what a failure mode looks like and how the reliability pattern responds to it. That behavior is deliberate — it is not a bug to "fix".


---
## 🔧 Part 1: Environment Setup

Load environment variables and import everything the rest of the notebook needs: standard library utilities for timing/retries, the LangGraph/LangChain building blocks, and LangSmith's `@traceable` decorator for tracing the fallback chain.


In [2]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and Environment Variables
# ============================================================================
import operator
import random
import time
from functools import wraps
from typing import Callable, Literal, Optional

from typing_extensions import Annotated, TypedDict

from dotenv import load_dotenv
from langsmith import traceable

# from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

# Load API keys (OPENAI_API_KEY, ANTHROPIC_API_KEY, LANGSMITH_API_KEY) from .env
load_dotenv()

print("✅ Environment loaded and imports ready.")


✅ Environment loaded and imports ready.


---
## 🔁 Part 2: Retry with Exponential Backoff

The simplest reliability pattern: when a call fails, wait a bit and try again, increasing the wait time (with random jitter) on each subsequent attempt so repeated retries don't all collide at once.

### Key Concepts:
- **Exponential backoff**: delay doubles each attempt, capped at `max_delay`.
- **Jitter**: a random multiplier on the delay so many concurrent callers don't retry in lockstep.


### `with_retry`
A decorator factory that wraps any function with retry-with-backoff logic. On each failure (matching `exceptions`), it sleeps for an exponentially growing, jittered delay before trying again; after `max_retries` attempts it re-raises the last exception.


In [3]:
# ============================================================================
# WITH_RETRY: Exponential Backoff Retry Decorator
# ============================================================================
def with_retry(
    max_retries: int = 3,
    base_delay: float = 1.0,
    max_delay: float = 30.0,
    exceptions: tuple = (Exception,),
):
    """Retry decorator with exponential backoff."""

    def decorator(func: Callable):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exception = e
                    if attempt < max_retries - 1:
                        delay = min(base_delay * (2**attempt), max_delay)
                        # Add jitter so retries from multiple callers don't collide
                        delay = delay * (0.5 + random.random())
                        print(
                            f"Attempt {attempt + 1} failed: {e}. Retrying in {delay:.1f}s..."
                        )
                        time.sleep(delay)
            raise last_exception

        return wrapper

    return decorator


print("✅ `with_retry` decorator defined.")


✅ `with_retry` decorator defined.


### `unreliable_api_call`
A stand-in for a flaky external API — it fails about half the time. Wrapping it with `@with_retry` is the intentional failure mode this demo showcases: the raised `ConnectionError` is simulated on purpose so you can watch the retry logic recover from it.


In [4]:
# ============================================================================
# UNRELIABLE_API_CALL: Simulated Flaky External API
# ============================================================================
@with_retry(max_retries=3, base_delay=1.0)
def unreliable_api_call(query: str) -> str:
    """Simulates an unreliable API. Fails ~50% of the time on purpose."""
    if random.random() < 0.5:
        raise ConnectionError("Simulated API failure")
    return f"Success: {query}"


### `demo_retry_pattern`
Runs a few queries through `unreliable_api_call` and prints whether each ultimately succeeded (after retries) or failed after exhausting all attempts.


In [5]:
# ============================================================================
# DEMO_RETRY_PATTERN: Exercise the Retry Decorator
# ============================================================================
def demo_retry_pattern():
    """Demonstrate retry with exponential backoff."""
    print("Retry Pattern Demo:\n")
    for i in range(3):
        try:
            result = unreliable_api_call(f"Query {i}")
            print(f"✅ {result}")
        except Exception as e:
            print(f"❌ Failed after retries: {e}")


---
## ⚡ Part 3: Circuit Breaker

Retries help with transient blips, but hammering a service that is *consistently* down wastes time and can make the outage worse. A circuit breaker tracks recent failures and, once a threshold is crossed, "opens" the circuit — short-circuiting calls immediately — until a recovery timeout has passed, at which point it lets one trial call through (`half-open`) to test recovery.

### Key Concepts:
- **closed**: normal operation, calls pass through.
- **open**: too many recent failures — calls are rejected immediately without hitting the real service.
- **half-open**: recovery timeout elapsed — the next call is a probe; success closes the circuit, failure re-opens it.


### `CircuitBreaker`
Implements the closed → open → half-open state machine described above around any callable.


In [6]:
# ============================================================================
# CIRCUITBREAKER: Circuit Breaker State Machine
# ============================================================================
class CircuitBreaker:
    """Circuit breaker pattern for failing services."""

    def __init__(self, failure_threshold: int = 5, recovery_timeout: float = 30.0):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.failures = 0
        self.last_failure_time = 0
        self.state = "closed"  # closed, open, half-open

    def call(self, func: Callable, *args, **kwargs):
        """Execute function with circuit breaker protection."""
        # Check if circuit should move from open to half-open
        if self.state == "open":
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = "half-open"
            else:
                raise Exception("Circuit breaker is OPEN")

        try:
            result = func(*args, **kwargs)
            # Success - reset on half-open
            if self.state == "half-open":
                self.state = "closed"
                self.failures = 0
            return result
        except Exception as e:
            self.failures += 1
            self.last_failure_time = time.time()
            if self.failures >= self.failure_threshold:
                self.state = "open"
            raise e


print("✅ `CircuitBreaker` class defined.")


✅ `CircuitBreaker` class defined.


### `demo_circuit_breaker`
Calls a service that fails 70% of the time on purpose, so the breaker trips open after `failure_threshold` failures. The demo then sleeps past `recovery_timeout` to show the breaker move to `half-open` and, on a successful probe, back to `closed`.


In [7]:
# ============================================================================
# DEMO_CIRCUIT_BREAKER: Exercise the Circuit Breaker States
# ============================================================================
def demo_circuit_breaker():
    """Demonstrate circuit breaker pattern."""
    breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=5.0)

    def flaky_service():
        # Intentionally fails 70% of the time to force the breaker open
        if random.random() < 0.7:
            raise Exception("Service error")
        return "OK"

    print("\nCircuit Breaker Demo:\n")
    for i in range(15):
        try:
            result = breaker.call(flaky_service)
            print(f"Attempt {i+1}: ✅ {result} (state: {breaker.state})")
        except Exception as e:
            print(f"Attempt {i+1}: ❌ {e} (state: {breaker.state})")

        # After attempt 7, wait long enough for recovery
        if i == 6:
            print("  ⏳ Waiting 6 seconds for recovery timeout...")
            time.sleep(6)
        else:
            time.sleep(0.5)


---
## 🔀 Part 4: Fallback Chains Across Models

When one provider or model is unavailable, a fallback chain tries the next one in a prioritized list instead of failing outright. This example also adds a simple response cache so a repeated query short-circuits straight to the cached answer instead of calling any model again.

### Key Concepts:
- **Fallback order**: cheapest/fastest model first, progressively more capable/expensive models as backups.
- **Caching**: successful responses are memoized by query so identical requests are free the second time.
- **Tracing**: `@traceable` reports the call to LangSmith so you can inspect which model actually served each query.


### `FallbackChain`
Tries `gpt-4o-mini` → `gpt-4o` → `claude-sonnet`, in order, returning the first successful response along with which model produced it. Only raises once every model in the chain has failed.


In [8]:
# ============================================================================
# FALLBACKCHAIN: Cascade Across Models with Caching
# ============================================================================
class FallbackChain:
    """Try multiple models in order until one succeeds."""

    def __init__(self):
        self.models = [
            ("gpt-4o-mini", ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=10)),
            ("gpt-4o", ChatOpenAI(model="gpt-4o", temperature=0, timeout=10)),
            (
                "claude-sonnet",
                ChatAnthropic(
                    model="claude-sonnet-4-5-20250929", temperature=0, timeout=10
                ),
            ),
        ]
        self.cache = {}

    @traceable(name="fallback_invoke")
    def invoke(self, query: str, use_cache: bool = True) -> tuple[str, str]:
        """
        Invoke with fallbacks.

        Returns: (response, model_used)
        """
        # Check cache first
        if use_cache and query in self.cache:
            return self.cache[query], "cache"

        errors = []
        for model_name, model in self.models:
            try:
                response = model.invoke(query)
                result = response.content
                # Cache successful response
                self.cache[query] = result
                return result, model_name
            except Exception as e:
                errors.append(f"{model_name}: {str(e)}")
                continue

        # All models failed
        raise Exception(f"All models failed: {errors}")


print("✅ `FallbackChain` class defined.")


✅ `FallbackChain` class defined.


### `demo_fallback_chain`
Sends a couple of distinct queries plus one repeat (to show the cache hit) through the chain, printing which model actually answered each one.


In [9]:
# ============================================================================
# DEMO_FALLBACK_CHAIN: Exercise the Fallback Chain and Cache
# ============================================================================
def demo_fallback_chain():
    """Demonstrate fallback chain."""
    chain = FallbackChain()

    print("\nFallback Chain Demo:\n")
    queries = [
        "What is 2 + 2?",
        "What is Python?",
        "What is 2 + 2?",  # Should hit cache
    ]

    for query in queries:
        try:
            result, model = chain.invoke(query)
            print(f"Query: {query}")
            print(f"  Model: {model}")
            print(f"  Response: {result[:50]}...")
        except Exception as e:
            print(f"Query: {query}")
            print(f"  ❌ Error: {e}")


---
## 🤖 Part 5: A Self-Healing LangGraph Agent

The previous patterns wrap individual function calls. This part pushes retry-and-report logic *into the graph itself*: a `process` node that can fail, a conditional edge that routes back to retry or forward to an error handler, and a dedicated `handle_error` node that turns a failure into a graceful user-facing message instead of an unhandled exception.

### Key Concepts:
- **State-tracked retries**: `retry_count` / `max_retries` live in graph state, so the routing function (not a Python loop) decides whether to retry.
- **Conditional edges as control flow**: `should_continue` returns one of `"retry"`, `"error"`, `"success"`, and the graph routes accordingly.


### `RobustState`
The graph state schema: chat messages, the last error (if any), how many retries have been used vs. allowed, and a success flag the routing function checks.


In [10]:
# ============================================================================
# ROBUSTSTATE: Graph State Schema
# ============================================================================
class RobustState(TypedDict):
    messages: Annotated[list, operator.add]
    error: Optional[str]
    retry_count: int
    max_retries: int
    success: bool


### `create_robust_agent`
Builds the graph: `process` simulates an occasional transient failure (intentionally, via `random.random() < 0.3`) and otherwise calls the LLM; `should_continue` inspects state to route to `process` again, to `handle_error`, or to `finalize`; `handle_error` converts a terminal failure into a friendly `AIMessage` instead of letting the exception propagate.


In [11]:
# ============================================================================
# CREATE_ROBUST_AGENT: Build the Self-Healing Graph
# ============================================================================
def create_robust_agent():
    """Create agent with built-in error handling."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    def process_with_retry(state: RobustState) -> dict:
        """Process with retry logic built-in."""
        try:
            # Simulate occasional failure (intentional, for the demo)
            if random.random() < 0.3 and state["retry_count"] < 2:
                raise Exception("Simulated processing error")
            response = llm.invoke(state["messages"])
            return {"messages": [response], "success": True, "error": None}
        except Exception as e:
            return {
                "error": str(e),
                "retry_count": state["retry_count"] + 1,
                "success": False,
            }

    def should_continue(state: RobustState) -> Literal["retry", "error", "success"]:
        if state["success"]:
            return "success"
        elif state["retry_count"] < state["max_retries"]:
            return "retry"
        else:
            return "error"

    def handle_error(state: RobustState) -> dict:
        return {
            "messages": [
                AIMessage(
                    content=f"I apologize, but I encountered an error: {state['error']}. "
                    "Please try again later."
                )
            ]
        }

    def finalize(state: RobustState) -> dict:
        return state

    # --- Build graph ---
    graph = StateGraph(RobustState)
    graph.add_node("process", process_with_retry)
    graph.add_node("handle_error", handle_error)
    graph.add_node("finalize", finalize)

    graph.add_edge(START, "process")
    graph.add_conditional_edges(
        "process",
        should_continue,
        {"retry": "process", "error": "handle_error", "success": "finalize"},
    )
    graph.add_edge("handle_error", END)
    graph.add_edge("finalize", END)

    return graph.compile()


print("✅ `create_robust_agent` graph builder defined.")


✅ `create_robust_agent` graph builder defined.


### `demo_robust_agent`
Runs the compiled graph three times from a fresh initial state, printing whether each run ultimately succeeded, how many retries it used, and the final response text.


In [12]:
# ============================================================================
# DEMO_ROBUST_AGENT: Exercise the Self-Healing Graph
# ============================================================================
def demo_robust_agent():
    """Demonstrate robust agent with error handling."""
    agent = create_robust_agent()

    print("\nRobust Agent Demo:\n")
    for i in range(3):
        result = agent.invoke(
            {
                "messages": [HumanMessage(content="Hello!")],
                "error": None,
                "retry_count": 0,
                "max_retries": 3,
                "success": False,
            }
        )
        status = "✅ Success" if result["success"] else "❌ Failed"
        print(f"Attempt {i+1}: {status}")
        print(f"  Retries used: {result['retry_count']}")
        print(f"  Response: {result['messages'][-1].content[:50]}...")


---
## ▶️ Part 6: Run a Demo

The original script guarded its demo calls behind `if __name__ == "__main__":`. In a notebook this still evaluates to `True`, so the cell below runs as-is. Only one demo is uncommented by default (`demo_robust_agent`) — uncomment any other line to run that pattern's demo instead.


In [13]:
# ============================================================================
# RUN: Execute a Demo
# ============================================================================
if __name__ == "__main__":
    # Example usage of the unreliable API call with retry logic
    # try:
    #     result = unreliable_api_call("Hello, World!")
    #     print(result)
    # except Exception as e:
    #     print(f"API call failed after retries: {e}")

    # Run the retry pattern demonstration
    # demo_retry_pattern()

    # Run the circuit breaker demonstration
    # demo_circuit_breaker()

    # Run the fallback chain demonstration
    # demo_fallback_chain()

    # Run the robust agent demonstration
    demo_robust_agent()



Robust Agent Demo:

Attempt 1: ✅ Success
  Retries used: 0
  Response: Hello! How can I assist you today?...
Attempt 2: ✅ Success
  Retries used: 0
  Response: Hello! How can I assist you today?...
Attempt 3: ✅ Success
  Retries used: 1
  Response: Hello! How can I assist you today?...


---
## 📝 Summary

### 1. Patterns Covered
- **Retry with backoff** (`with_retry`, `unreliable_api_call`, `demo_retry_pattern`): recover from transient failures with exponentially growing, jittered delays.
- **Circuit breaker** (`CircuitBreaker`, `demo_circuit_breaker`): stop calling a service that is persistently failing, and probe for recovery after a timeout.
- **Fallback chain with caching** (`FallbackChain`, `demo_fallback_chain`): cascade across models when one is unavailable, and cache successful responses.
- **Self-healing graph** (`RobustState`, `create_robust_agent`, `demo_robust_agent`): push retry/error-handling logic into LangGraph state and conditional edges so failures produce a graceful message instead of an unhandled exception.

### 2. When to Use Which
- **Retry** — for calls that fail intermittently and independently (network blips).
- **Circuit breaker** — for calls to a *dependency* that can be persistently down; avoids piling on load during an outage.
- **Fallback chain** — when you have multiple interchangeable providers/models and want graceful degradation instead of hard failure.
- **Graph-level handling** — when the retry/error logic needs to be part of the agent's own control flow and visible in its state (e.g., for observability or user-facing messaging).

### Next Steps
- Continue to the next notebook in `03_LangGraph_Fundamentals/` to build on these mechanics.
- Consider combining patterns — e.g., wrap each leg of a `FallbackChain` in its own `CircuitBreaker`.
